# Muscle Analysis: CMC for BAA Baseline vs LGS 24w

Scale each session using the bilateral hindlimb muscle model from the RatHindlimb repository, then run CMC using existing IK kinematics and force plate data.

In [ ]:
import re
from pathlib import Path
import opensim as osim
import ezc3d
import polars as pl
import matplotlib.pyplot as plt

from osimpy import CMCSettings, sto_to_df
from rathindlimb.scale_utils import (
    RatScalingParameters,
    scale_opensim_model,
    scaling_parameters_from_c3d,
)

# --- Paths to generic model and setup files ---
MODELS_DIR = RATHINDLIMB_ROOT / "models" / "osim"
UNSCALED_MODEL = MODELS_DIR / "rat_hindlimb_bilateral.osim"
MARKER_SET = MODELS_DIR / "rat_hindlimb_bilateral_markerset.xml"
SCALE_SETUP = MODELS_DIR / "rat_hindlimb_bilateral_scale_setup.xml"
TASK_SET = MODELS_DIR / "rat_hindlimb_bilateral_taskSet.xml"
CONTROL_CONSTRAINTS = MODELS_DIR / "rat_hindlimb_bilateral_controlconstraints.xml"
ACTUATORS = MODELS_DIR / "rat_hindlimb_bilateral_actuators.xml"

DATA_ROOT = Path("/home/hudson/Desktop/HudsonVault/Quarto/thesis/data/mocap")

print("Imports OK")

Imports OK


In [ ]:
# --- Session definitions ---
# BAA animals: Baseline (healthy controls)
# LGS animals: 24w post-VML injury
SESSIONS: dict[str, list[tuple[str, str]]] = {
    "Baseline": [
        ("BAA01", "Baseline"),
        ("BAA02", "Baseline"),
        ("BAA03", "Baseline"),
    ],
    "24w": [
        ("LGS11", "24w"),
        ("LGS12", "24w"),
        ("LGS14", "24w"),
    ],
}

def discover_trials(animal: str, session: str) -> list[dict[str, Path]]:
    """Find walk trials with complete IK + ID + FP data (prefixed naming)."""
    session_dir = DATA_ROOT / animal / session
    prefix = f"{animal}_{session}"
    trials = []
    for ik_file in sorted(session_dir.glob(f"{prefix}_Walk*_ik.mot")):
        trial_stem = ik_file.stem.replace("_ik", "")  # e.g. BAA01_Baseline_Walk05
        id_file = session_dir / f"{trial_stem}_id.sto"
        fp_mot = session_dir / f"{trial_stem}_FP.mot"
        fp_xml = session_dir / f"{trial_stem}_fp_setup.xml"
        if id_file.exists() and fp_mot.exists() and fp_xml.exists():
            trials.append({
                "name": trial_stem,
                "session_dir": session_dir,
                "ik_file": ik_file,
                "id_file": id_file,
                "fp_mot": fp_mot,
                "fp_xml": fp_xml,
            })
    return trials

# Verify all sessions
for group, sessions in SESSIONS.items():
    print(f"\n--- {group} ---")
    for animal, session in sessions:
        trials = discover_trials(animal, session)
        walk_names = [t["name"].split("_")[-1] for t in trials]
        print(f"  {animal}/{session}: {len(trials)} trials ({', '.join(walk_names)})")


--- Baseline ---
  BAA01/Baseline: 5 trials (Walk05, Walk06, Walk08, Walk11, Walk14)
  BAA02/Baseline: 5 trials (Walk04, Walk08, Walk10, Walk11, Walk12)
  BAA03/Baseline: 4 trials (Walk08, Walk09, Walk10, Walk11)

--- 24w ---
  LGS09/24w: 3 trials (Walk04, Walk06, Walk16)
  LGS10/24w: 3 trials (Walk07, Walk08, Walk14)
  LGS14/24w: 7 trials (Walk01, Walk02, Walk04, Walk05, Walk06, Walk07, Walk10)


## Scale Models

Scale the bilateral hindlimb muscle model for each session using the static C3D trial.

In [3]:
def sanitize_model_name(model_path: Path) -> None:
    """Ensure the <Model name=...> tag doesn't contain path separators."""
    expected_name = model_path.stem
    try:
        model = osim.Model(str(model_path))
        if str(model.getName()) != expected_name:
            model.setName(expected_name)
            model.printToXML(str(model_path))
    except Exception:
        content = model_path.read_text(encoding="utf-8")
        updated = re.sub(
            r'<Model name="[^"]+">',
            f'<Model name="{expected_name}">',
            content,
            count=1,
        )
        model_path.write_text(updated, encoding="utf-8")


def scale_session(animal: str, session: str) -> Path:
    """Scale the bilateral model for a session. Returns path to the scaled model (with muscles)."""
    session_dir = DATA_ROOT / animal / session
    prefix = f"{animal}_{session}"

    # Use the last Static C3D (highest number = usually best quality)
    static_c3ds = sorted(session_dir.glob("Static*.c3d"))
    if not static_c3ds:
        raise FileNotFoundError(f"No Static C3D files in {session_dir}")
    static_c3d = static_c3ds[-1]

    # Find or create the TRC
    trc_name = f"{prefix}_{static_c3d.stem}"
    trc_path = session_dir / f"{trc_name}.trc"
    if not trc_path.exists():
        # Check if existing TRC with old naming exists
        old_trc = session_dir / f"{prefix}_{static_c3d.stem}.trc"
        if old_trc.exists():
            trc_path = old_trc
        else:
            adapter = osim.C3DFileAdapter()
            tables = adapter.read(str(static_c3d))
            marker_table = adapter.getMarkersTable(tables)
            osim.TRCFileAdapter.write(marker_table, str(trc_path))
            print(f"  Created TRC: {trc_path.name}")

    # Output name for the new muscle model scale
    scale_name = f"{prefix}_muscle"
    scaled_model_path = session_dir / f"{scale_name}_scaled.osim"

    if scaled_model_path.exists():
        print(f"  Scaled model already exists: {scaled_model_path.name}")
        return scaled_model_path

    # Extract scaling parameters from static C3D
    scale_params = scaling_parameters_from_c3d(str(static_c3d))

    # Get time range from static TRC
    c3d_data = ezc3d.c3d(str(static_c3d), extract_forceplat_data=False)
    point_rate = float(c3d_data.parameters["POINT"]["RATE"]["value"][0])
    first_frame = int(c3d_data.header["points"]["first_frame"])
    last_frame = int(c3d_data.header["points"]["last_frame"])
    time_start = first_frame / point_rate
    time_end = last_frame / point_rate

    # Run scaling
    scaled_path, marker_path, setup_path, factors_path = scale_opensim_model(
        name=scale_name,
        unscaled_model_path=str(UNSCALED_MODEL),
        marker_set_path=str(MARKER_SET),
        marker_file_name=trc_path.name,
        parameters=scale_params,
        output_dir=str(session_dir),
        scale_setup_path=str(SCALE_SETUP),
        time_start=time_start,
        time_end=time_end,
    )

    scaled_model_path = Path(scaled_path)
    marker_model_path = Path(marker_path)
    sanitize_model_name(scaled_model_path)
    sanitize_model_name(marker_model_path)

    print(f"  Scaled model created: {scaled_model_path.name}")
    return scaled_model_path


# Scale all sessions
scaled_models: dict[str, Path] = {}
for group, sessions in SESSIONS.items():
    for animal, session in sessions:
        key = f"{animal}_{session}"
        print(f"Scaling {key}...")
        try:
            scaled_models[key] = scale_session(animal, session)
        except Exception as e:
            print(f"  ERROR: {e}")
            scaled_models[key] = None

print("\n--- Scaled models ---")
for key, path in scaled_models.items():
    status = path.name if path else "FAILED"
    print(f"  {key}: {status}")

Scaling BAA01_Baseline...
  Scaled model already exists: BAA01_Baseline_muscle_scaled.osim
Scaling BAA02_Baseline...
  Scaled model already exists: BAA02_Baseline_muscle_scaled.osim
Scaling BAA03_Baseline...
  Scaled model already exists: BAA03_Baseline_muscle_scaled.osim
Scaling LGS09_24w...
  Scaled model already exists: LGS09_24w_muscle_scaled.osim
Scaling LGS10_24w...
  Scaled model already exists: LGS10_24w_muscle_scaled.osim
Scaling LGS14_24w...
  Scaled model already exists: LGS14_24w_muscle_scaled.osim

--- Scaled models ---
  BAA01_Baseline: BAA01_Baseline_muscle_scaled.osim
  BAA02_Baseline: BAA02_Baseline_muscle_scaled.osim
  BAA03_Baseline: BAA03_Baseline_muscle_scaled.osim
  LGS09_24w: LGS09_24w_muscle_scaled.osim
  LGS10_24w: LGS10_24w_muscle_scaled.osim
  LGS14_24w: LGS14_24w_muscle_scaled.osim


## Fix External Loads Paths & Run CMC

Some external loads XML files contain hardcoded Windows paths. Fix them to use relative filenames, then run CMC for each trial.

In [ ]:
import subprocess
import json
import time as _time
from pathlib import Path


def run_cmc_trial(
    model_file: Path,
    trial: dict[str, Path],
    output_dir: Path | None = None,
    timeout: int = 900,
) -> dict:
    """Run CMC in an isolated subprocess to prevent kernel crashes.

    Each trial runs in its own process — if OpenSim segfaults or OOMs,
    only that subprocess dies, not the notebook kernel.
    """
    trial_name = trial["name"]
    session_dir = trial["session_dir"]
    ik_file = trial["ik_file"]
    fp_xml = trial["fp_xml"]

    if output_dir is None:
        output_dir = session_dir / "cmc_results" / trial_name

    # Write a log file per trial for debugging
    log_dir = session_dir / "cmc_results"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f"{trial_name}_cmc.log"

    cmd = [
        PYTHON, str(CMC_SCRIPT),
        str(model_file), str(ik_file), str(fp_xml), str(output_dir),
    ]

    t0 = _time.time()
    try:
        with open(log_file, "w") as lf:
            proc = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,
                stderr=lf,
                text=True,
                timeout=timeout,
                cwd=str(session_dir),
            )
        elapsed = _time.time() - t0

        # Parse the JSON result from the last CMC_RESULT line
        for line in reversed(proc.stdout.splitlines()):
            if line.startswith("CMC_RESULT:"):
                result = json.loads(line[len("CMC_RESULT:"):])
                result["trial"] = trial_name
                if result["success"]:
                    result["controls_file"] = Path(result["controls_file"])
                    result["forces_file"] = Path(result["forces_file"])
                    result["states_file"] = Path(result["states_file"])
                    print(f"    ✓ succeeded (attempt: {result['attempt']}, {elapsed:.0f}s)")
                else:
                    print(f"    ✗ all attempts failed ({elapsed:.0f}s)")
                return result

        # No JSON result found — process crashed (segfault, OOM, etc.)
        print(f"    ✗ process crashed (code {proc.returncode}, {elapsed:.0f}s). See {log_file.name}")
        return {"trial": trial_name, "success": False, "error": f"crash (exit {proc.returncode})"}

    except subprocess.TimeoutExpired:
        print(f"    ✗ timed out after {timeout}s")
        return {"trial": trial_name, "success": False, "error": "timeout"}
    except Exception as e:

        print(f"    ✗ error: {e}")
        return {"trial": trial_name, "success": False, "error": str(e)}

print("CMC subprocess runner ready")

CMC subprocess runner ready


In [5]:
# --- Run CMC for all sessions ---
cmc_results: dict[str, list[dict]] = {}

for group, sessions in SESSIONS.items():
    for animal, session in sessions:
        key = f"{animal}_{session}"
        model_path = scaled_models.get(key)
        if model_path is None:
            print(f"Skipping {key}: no scaled model")
            continue

        trials = discover_trials(animal, session)
        print(f"\n{'='*60}")
        print(f"CMC for {key} ({len(trials)} trials) using {model_path.name}")
        print(f"{'='*60}")

        session_results = []
        for trial in trials:
            walk = trial["name"].split("_")[-1]
            print(f"  {walk}:")

            # Check if CMC results already exist
            cmc_dir = trial["session_dir"] / "cmc_results" / trial["name"]
            controls_candidates = list(cmc_dir.glob("*controls*.sto")) if cmc_dir.exists() else []
            if controls_candidates:
                print(f"    ✓ CMC results already exist, reusing")
                forces = list(cmc_dir.glob("*Actuation_force*.sto"))
                states = list(cmc_dir.glob("*states*.sto"))
                session_results.append({
                    "trial": trial["name"],
                    "attempt": "existing",
                    "controls_file": controls_candidates[0],
                    "forces_file": forces[0] if forces else None,
                    "states_file": states[0] if states else None,
                    "success": True,
                })
                continue

            result = run_cmc_trial(model_path, trial)
            session_results.append(result)

        cmc_results[key] = session_results

# Summary
print(f"\n{'='*60}")
print("CMC Summary")
print(f"{'='*60}")
for key, results in cmc_results.items():
    successes = sum(1 for r in results if r.get("success"))
    total = len(results)
    print(f"  {key}: {successes}/{total} trials succeeded")


CMC for BAA01_Baseline (5 trials) using BAA01_Baseline_muscle_scaled.osim
  Walk05:


KeyboardInterrupt: 

## Compare Muscle Activations: Baseline vs 24w

Load CMC controls (muscle activations) from successful trials and compare BAA (Baseline) vs LGS (24w).

In [ ]:
def load_cmc_controls(result: dict) -> pl.DataFrame | None:
    """Load CMC controls file and add metadata columns."""
    if not result.get("success") or result.get("controls_file") is None:
        return None
    try:
        df, _ = sto_to_df(str(result["controls_file"]))
        trial_name = result["trial"]
        parts = trial_name.split("_")
        animal = parts[0]
        session = parts[1]
        walk = parts[2] if len(parts) > 2 else "unknown"
        df = df.with_columns(
            pl.lit(animal).alias("animal"),
            pl.lit(session).alias("session"),
            pl.lit(walk).alias("walk"),
            pl.lit(trial_name).alias("trial"),
        )
        return df
    except Exception as e:
        print(f"  Error loading {result['trial']}: {e}")
        return None


def load_cmc_forces(result: dict) -> pl.DataFrame | None:
    """Load CMC actuation force file and add metadata columns."""
    if not result.get("success") or result.get("forces_file") is None:
        return None
    try:
        df, _ = sto_to_df(str(result["forces_file"]))
        trial_name = result["trial"]
        parts = trial_name.split("_")
        df = df.with_columns(
            pl.lit(parts[0]).alias("animal"),
            pl.lit(parts[1]).alias("session"),
            pl.lit(parts[2] if len(parts) > 2 else "unknown").alias("walk"),
            pl.lit(trial_name).alias("trial"),
        )
        return df
    except Exception as e:
        print(f"  Error loading forces for {result['trial']}: {e}")
        return None


# Collect all successful results
all_controls: list[pl.DataFrame] = []
all_forces: list[pl.DataFrame] = []

for key, results in cmc_results.items():
    for result in results:
        ctrl_df = load_cmc_controls(result)
        if ctrl_df is not None:
            all_controls.append(ctrl_df)
        force_df = load_cmc_forces(result)
        if force_df is not None:
            all_forces.append(force_df)

if all_controls:
    controls_df = pl.concat(all_controls, how="diagonal_relaxed")
    print(f"Controls: {controls_df.shape[0]} rows from {controls_df['trial'].n_unique()} trials")
    print(f"Animals: {controls_df['animal'].unique().to_list()}")
    print(f"Sessions: {controls_df['session'].unique().to_list()}")
else:
    print("No CMC controls data loaded")
    controls_df = None

if all_forces:
    forces_df = pl.concat(all_forces, how="diagonal_relaxed")
    print(f"\nForces: {forces_df.shape[0]} rows from {forces_df['trial'].n_unique()} trials")
else:
    print("No CMC forces data loaded")
    forces_df = None

In [ ]:
if controls_df is not None:
    # Identify muscle columns (exclude metadata and time)
    meta_cols = {"time", "animal", "session", "walk", "trial"}
    muscle_cols = [c for c in controls_df.columns if c not in meta_cols]

    # Pick muscles with highest variance across all trials (most informative)
    variances = {}
    for col in muscle_cols:
        try:
            v = controls_df[col].drop_nulls().var()
            if v is not None:
                variances[col] = v
        except Exception:
            pass

    top_muscles = sorted(variances, key=variances.get, reverse=True)[:12]
    print(f"Top 12 muscles by variance: {top_muscles}")

    # Group by session (Baseline vs 24w)
    baseline_df = controls_df.filter(pl.col("session") == "Baseline")
    injury_df = controls_df.filter(pl.col("session") == "24w")

    n_muscles = len(top_muscles)
    n_cols = 3
    n_rows = (n_muscles + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows), sharex=False)
    axes = axes.flatten()

    for idx, muscle in enumerate(top_muscles):
        ax = axes[idx]
        # Plot individual Baseline trials (blue, thin)
        for trial_name in baseline_df["trial"].unique().to_list():
            trial_data = baseline_df.filter(pl.col("trial") == trial_name)
            if muscle in trial_data.columns:
                vals = trial_data[muscle].drop_nulls().to_numpy()
                t = trial_data["time"].drop_nulls().to_numpy()[:len(vals)]
                if len(vals) > 0:
                    ax.plot(t, vals, color="tab:blue", alpha=0.3, linewidth=0.8)

        # Plot individual 24w trials (red, thin)
        for trial_name in injury_df["trial"].unique().to_list():
            trial_data = injury_df.filter(pl.col("trial") == trial_name)
            if muscle in trial_data.columns:
                vals = trial_data[muscle].drop_nulls().to_numpy()
                t = trial_data["time"].drop_nulls().to_numpy()[:len(vals)]
                if len(vals) > 0:
                    ax.plot(t, vals, color="tab:red", alpha=0.3, linewidth=0.8)

        ax.set_title(muscle, fontsize=9)
        ax.set_ylabel("Activation")
        ax.set_xlabel("Time (s)")

    # Remove empty subplots
    for idx in range(len(top_muscles), len(axes)):
        fig.delaxes(axes[idx])

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="tab:blue", label="Baseline (BAA)"),
        Line2D([0], [0], color="tab:red", label="24w (LGS)"),
    ]
    fig.legend(handles=legend_elements, loc="upper right", fontsize=10)
    fig.suptitle("CMC Muscle Activations: Baseline vs 24w Post-VML", fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
else:
    print("No controls data available to plot")

In [ ]:
if forces_df is not None:
    meta_cols = {"time", "animal", "session", "walk", "trial"}
    force_cols = [c for c in forces_df.columns if c not in meta_cols]

    # Pick top muscles by variance in force output
    variances = {}
    for col in force_cols:
        try:
            v = forces_df[col].drop_nulls().var()
            if v is not None:
                variances[col] = v
        except Exception:
            pass

    top_force_muscles = sorted(variances, key=variances.get, reverse=True)[:12]

    baseline_f = forces_df.filter(pl.col("session") == "Baseline")
    injury_f = forces_df.filter(pl.col("session") == "24w")

    n_muscles = len(top_force_muscles)
    n_cols = 3
    n_rows = (n_muscles + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows), sharex=False)
    axes = axes.flatten()

    for idx, muscle in enumerate(top_force_muscles):
        ax = axes[idx]
        for trial_name in baseline_f["trial"].unique().to_list():
            trial_data = baseline_f.filter(pl.col("trial") == trial_name)
            if muscle in trial_data.columns:
                vals = trial_data[muscle].drop_nulls().to_numpy()
                t = trial_data["time"].drop_nulls().to_numpy()[:len(vals)]
                if len(vals) > 0:
                    ax.plot(t, vals, color="tab:blue", alpha=0.3, linewidth=0.8)

        for trial_name in injury_f["trial"].unique().to_list():
            trial_data = injury_f.filter(pl.col("trial") == trial_name)
            if muscle in trial_data.columns:
                vals = trial_data[muscle].drop_nulls().to_numpy()
                t = trial_data["time"].drop_nulls().to_numpy()[:len(vals)]
                if len(vals) > 0:
                    ax.plot(t, vals, color="tab:red", alpha=0.3, linewidth=0.8)

        ax.set_title(muscle, fontsize=9)
        ax.set_ylabel("Force (N)")
        ax.set_xlabel("Time (s)")

    for idx in range(len(top_force_muscles), len(axes)):
        fig.delaxes(axes[idx])

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="tab:blue", label="Baseline (BAA)"),
        Line2D([0], [0], color="tab:red", label="24w (LGS)"),
    ]
    fig.legend(handles=legend_elements, loc="upper right", fontsize=10)
    fig.suptitle("CMC Muscle Forces: Baseline vs 24w Post-VML", fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
else:
    print("No forces data available to plot")